# CIFAR-100 — Wide ResNet-16-4, thử 20 epoch

Notebook này tự chứa toàn bộ mã train, không cần file Python bên ngoài. Model được khởi tạo ngẫu nhiên, không dùng trọng số pretrained. Dữ liệu đọc từ thư mục `cifar-100` cạnh project.

Augmentation train theo `minidl-cifar100-3.ipynb`: crop ngẫu nhiên 32×32 sau khi pad 4 pixel bằng 0, rồi lật ngang với xác suất 50%. Ảnh giữ kích thước 32×32 cho WRN và được chuẩn hóa bằng mean/std của tập train. Validation và test không dùng augmentation.


## Cấu hình và đọc dữ liệu


In [ ]:
from __future__ import annotations

import math
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_DIR = next(
    (path.resolve() for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "fruit-detection")
     if (path / "cifar-100" / "train").is_file()),
    Path.cwd().resolve(),
)
IMAGE_SIZE = 32
NUM_CLASSES = 100

@dataclass(frozen=True)
class TrainConfig:
    epochs: int = 20
    batch_size: int = 64
    seed: int = 42
    validation_size: int = 5_000

def load_cifar100(data_dir: Path = PROJECT_DIR / "cifar-100"):
    def load_batch(path: Path):
        with path.open("rb") as stream:
            data = pickle.load(stream, encoding="bytes")
        images = np.asarray(data[b"data"], dtype=np.uint8).reshape(-1, 3, 32, 32)
        images = images.transpose(0, 2, 3, 1).copy()
        labels = np.asarray(data[b"fine_labels"], dtype=np.int32)
        return images, labels

    required = (data_dir / "train", data_dir / "test", data_dir / "meta")
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Missing local CIFAR-100 files: " + ", ".join(missing))
    train_images, train_labels = load_batch(data_dir / "train")
    test_images, test_labels = load_batch(data_dir / "test")
    return train_images, train_labels, test_images, test_labels


## Augmentation train

Như `minidl-cifar100-3.ipynb`, augmentation chỉ gồm `RandomCrop(32, padding=4)` với padding 0 và `RandomHorizontalFlip(p=0.5)`. Chuẩn hóa diễn ra sau augmentation. WRN học từ đầu trên ảnh 32×32, nên không resize lên 224×224 và dùng mean/std tính từ 45.000 ảnh train thay vì thống kê ImageNet.


## Pipeline train và validation

Mean/std tính riêng trên 45.000 ảnh train. Train dùng nhãn one-hot cho loss categorical; validation dùng cùng chuẩn hóa nhưng không crop hoặc lật ảnh.


In [ ]:
def augment_image(image: tf.Tensor, mean: tf.Tensor, std: tf.Tensor) -> tf.Tensor:
    """Apply the reference notebook's crop and horizontal flip at 32x32."""
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.pad(image, [[4, 4], [4, 4], [0, 0]])  # RandomCrop padding=4 uses zero fill.
    image = tf.image.random_crop(image, [IMAGE_SIZE, IMAGE_SIZE, 3])
    image = tf.image.random_flip_left_right(image)
    return (image - mean) / std


def make_datasets(config: TrainConfig):
    train_images, train_labels, _, _ = load_cifar100()
    generator = np.random.default_rng(config.seed)
    permutation = generator.permutation(len(train_images))
    validation_indices = permutation[: config.validation_size]
    training_indices = permutation[config.validation_size :]
    train_x, train_y = train_images[training_indices], train_labels[training_indices]
    val_x, val_y = train_images[validation_indices], train_labels[validation_indices]

    mean = tf.constant(train_x.mean(axis=(0, 1, 2), dtype=np.float64) / 255.0, tf.float32)
    std = tf.constant(train_x.std(axis=(0, 1, 2), dtype=np.float64) / 255.0, tf.float32)
    autotune = tf.data.AUTOTUNE

    def augment(image, label):
        return augment_image(image, mean, std), tf.one_hot(label, NUM_CLASSES)

    def clean(image, label):
        image = (tf.cast(image, tf.float32) / 255.0 - mean) / std
        return image, tf.one_hot(label, NUM_CLASSES)

    train_ds = (
        tf.data.Dataset.from_tensor_slices((train_x, train_y))
        .shuffle(len(train_x), seed=config.seed + 1, reshuffle_each_iteration=True)
        .map(augment, num_parallel_calls=autotune)
        .batch(config.batch_size)
        .prefetch(autotune)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((val_x, val_y))
        .map(clean, num_parallel_calls=autotune)
        .batch(config.batch_size)
        .prefetch(autotune)
    )
    return train_ds, val_ds, mean, std


## Preview: ảnh gốc và ảnh sau augmentation

Chạy các cell từ đầu notebook đến đây; **không cần chạy cell train**. Mỗi cột là cùng một ảnh trước/sau crop ngẫu nhiên và lật ngang. Ảnh sau chuẩn hóa được đổi ngược về dải `[0, 1]` chỉ để hiển thị.


In [ ]:
import matplotlib.pyplot as plt

preview_config = TrainConfig()
all_train_images, all_train_labels, _, _ = load_cifar100()
train_indices = np.random.default_rng(preview_config.seed).permutation(len(all_train_images))[
    preview_config.validation_size:
]
train_images_for_preview = all_train_images[train_indices]
train_labels_for_preview = all_train_labels[train_indices]
preview_mean = train_images_for_preview.mean(axis=(0, 1, 2), dtype=np.float64) / 255.0
preview_std = train_images_for_preview.std(axis=(0, 1, 2), dtype=np.float64) / 255.0
preview_ids = np.random.default_rng(7).choice(len(train_indices), size=5, replace=False)
preview_images = train_images_for_preview[preview_ids].copy()
preview_labels = train_labels_for_preview[preview_ids].copy()
with (PROJECT_DIR / "cifar-100" / "meta").open("rb") as stream:
    class_names = [name.decode("utf-8") for name in pickle.load(stream, encoding="bytes")[b"fine_label_names"]]
del all_train_images, all_train_labels, train_images_for_preview, train_labels_for_preview

mean_tf = tf.constant(preview_mean, tf.float32)
std_tf = tf.constant(preview_std, tf.float32)

def denormalize(image):
    return np.clip(np.asarray(image) * preview_std + preview_mean, 0.0, 1.0)

augmented = [augment_image(tf.convert_to_tensor(image), mean_tf, std_tf)
             for image in preview_images]
fig, axes = plt.subplots(2, len(preview_images), figsize=(15, 6))
for index, (original, processed, label) in enumerate(zip(preview_images, augmented, preview_labels)):
    axes[0, index].imshow(original)
    axes[0, index].set_title(f"Gốc: {class_names[int(label)]}", fontsize=10)
    axes[1, index].imshow(denormalize(processed.numpy()))
    axes[1, index].set_title("Sau augment", fontsize=10)
    axes[0, index].axis("off")
    axes[1, index].axis("off")
fig.suptitle("Cùng ảnh trước và sau augmentation từng ảnh", fontsize=15)
fig.tight_layout()
preview_dir = PROJECT_DIR / "artifacts" / "plots"
preview_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(preview_dir / "wrn_before_after_augmentation.png", dpi=180, bbox_inches="tight")
plt.show()

## Wide ResNet-16-4 từ đầu

Hai residual block cho mỗi stage, độ rộng 64–128–256, kết thúc bằng global average pooling.


In [ ]:
def build_wrn_16_4() -> tf.keras.Model:
    """CIFAR-style Wide ResNet with two residual blocks per stage."""
    layers = tf.keras.layers

    def block(inputs, channels: int, stride: int):
        x = layers.BatchNormalization()(inputs)
        x = layers.Activation("relu")(x)
        shortcut = inputs
        if stride != 1 or inputs.shape[-1] != channels:
            shortcut = layers.Conv2D(channels, 1, strides=stride, padding="same", use_bias=False)(inputs)
        x = layers.Conv2D(channels, 3, strides=stride, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
        return layers.Add()([shortcut, x])

    inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    x = layers.Conv2D(16, 3, padding="same", use_bias=False)(inputs)
    for channels, first_stride in ((64, 1), (128, 2), (256, 2)):
        x = block(x, channels, first_stride)
        x = block(x, channels, 1)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="wrn_16_4_cifar100")

## Chạy thử 20 epoch

Checkpoint lưu theo `val_accuracy` tốt nhất. Dùng validation accuracy/loss để đánh giá; tập test chưa được sử dụng trong lượt thử.


In [ ]:
def run_experiment(
    config: TrainConfig = TrainConfig(),
    *,
    smoke_test: bool = False,
    use_plateau_callbacks: bool = False,
):
    tf.keras.utils.set_random_seed(config.seed)
    train_ds, val_ds, mean, std = make_datasets(config)
    model = build_wrn_16_4()
    sample_images, sample_labels = next(iter(train_ds))
    assert sample_images.shape[1:] == (32, 32, 3)
    assert sample_labels.shape[-1] == NUM_CLASSES
    tf.debugging.assert_near(tf.reduce_sum(sample_labels, axis=1), tf.ones(tf.shape(sample_labels)[0]))
    tf.debugging.assert_all_finite(sample_images, "Augmentation produced non-finite images")

    steps_per_epoch = math.ceil((50_000 - config.validation_size) / config.batch_size)
    if use_plateau_callbacks:
        learning_rate = 1e-3
    else:
        learning_rate = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=1e-3,
            decay_steps=steps_per_epoch * config.epochs,
            alpha=0.01,
        )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
    )
    print(f"Model parameters: {model.count_params():,}", flush=True)
    print(f"Train batches: {steps_per_epoch}; validation batches: {len(val_ds)}", flush=True)
    print(f"Training RGB mean: {mean.numpy()}; std: {std.numpy()}", flush=True)

    if smoke_test:
        result = model.train_on_batch(sample_images, sample_labels, return_dict=True)
        print(f"One-batch smoke test: {result}", flush=True)
        return result

    checkpoint_dir = PROJECT_DIR / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    checkpoint_path = checkpoint_dir / f"cifar100_wrn16_4_aug{config.epochs}_best.keras"
    history_path = checkpoint_dir / f"cifar100_wrn16_4_aug{config.epochs}_history.csv"
    if use_plateau_callbacks:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(checkpoint_path),
                monitor="val_loss",
                mode="min",
                save_best_only=True,
                verbose=1,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=3,
                min_delta=0.0,
                min_lr=1e-5,
                verbose=1,
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=20,
                restore_best_weights=True,
                verbose=1,
            ),
            tf.keras.callbacks.CSVLogger(str(history_path), append=False),
        ]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                str(checkpoint_path), monitor="val_accuracy", mode="max", save_best_only=True
            ),
            tf.keras.callbacks.CSVLogger(str(history_path)),
        ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.epochs,
        callbacks=callbacks,
        verbose=2,
    )
    best_model = tf.keras.models.load_model(checkpoint_path)
    val_metrics = best_model.evaluate(val_ds, return_dict=True, verbose=0)
    print(f"Best checkpoint: {checkpoint_path}", flush=True)
    print(f"History: {history_path}", flush=True)
    print(f"Best validation metrics: {val_metrics}", flush=True)
    return history, val_metrics


In [ ]:
config = TrainConfig(epochs=20)
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("Data:", PROJECT_DIR / "cifar-100")
model = build_wrn_16_4()
print("Parameters:", f"{model.count_params():,}")
model.summary()


In [ ]:
history, val_metrics = run_experiment(config)
print("Validation của checkpoint tốt nhất:", val_metrics)


## Train tối đa 200 epoch

Chạy sau lượt thử 20 epoch nếu muốn huấn luyện đầy đủ. Cấu hình callback theo `build_cnn_cifar_100`: lưu checkpoint tốt nhất theo `val_loss`, giảm learning rate khi validation chững lại, dừng sớm sau 20 epoch không cải thiện và ghi CSV history. WRN vẫn dùng AdamW với weight decay `1e-4`; checkpoint 200 epoch được lưu riêng.


In [ ]:
config_200 = TrainConfig(epochs=200)
history_200, val_metrics_200 = run_experiment(
    config_200,
    use_plateau_callbacks=True,
)
print("Validation của checkpoint 200 epoch tốt nhất:", val_metrics_200)


## Đánh giá checkpoint trên test set

Đánh giá checkpoint có `val_loss` tốt nhất của lượt train 200 epoch trên 10.000 ảnh test CIFAR-100. Dùng lại mean/std của đúng 45.000 ảnh train và chỉ chuẩn hóa ảnh, không augmentation. Test set chỉ được dùng ở bước đánh giá cuối.


In [ ]:
checkpoint_200_path = PROJECT_DIR / "checkpoints" / "cifar100_wrn16_4_aug200_best.keras"
if not checkpoint_200_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_200_path}")

test_config = TrainConfig(epochs=200)
train_images, _, test_images, test_labels = load_cifar100()
split_rng = np.random.default_rng(test_config.seed)
train_indices = split_rng.permutation(len(train_images))[test_config.validation_size:]
train_images_for_stats = train_images[train_indices]
train_mean = train_images_for_stats.mean(axis=(0, 1, 2), dtype=np.float64) / 255.0
train_std = train_images_for_stats.std(axis=(0, 1, 2), dtype=np.float64) / 255.0
del train_images, train_images_for_stats

def normalize_test_image(image, label):
    image = (tf.cast(image, tf.float32) / 255.0 - train_mean) / train_std
    return image, tf.one_hot(label, NUM_CLASSES)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    .map(normalize_test_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(test_config.batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
best_model_200 = tf.keras.models.load_model(str(checkpoint_200_path))
test_metrics = best_model_200.evaluate(test_ds, return_dict=True, verbose=1)
print(f"Test images: {len(test_labels):,}")
print("CIFAR-100 test metrics:", test_metrics)
